<div style="background-color: #f0f4ff; padding: 16px 20px; border-left: 5px solid #4a6fa5; border-radius: 4px;">
<h2 style="margin: 0 0 8px 0;">Lab 2: Structured Data Extraction</h2>
<p style="margin: 0 0 16px 0; font-size: 13px; color: #555;">Adapted from Chan, Yee Kit, Erica Lai, and Yu Chen. "Teaching Tip: Teaching Undergraduate IS Students Hands-on Generative AI Development Skills." <em>Journal of Information Systems Education</em> 37(1), 2026.</p>
<hr style="border: none; border-top: 1px solid #ccc; margin-bottom: 16px;">
<b>First time in a coding environment?</b> You do not need to write any code — just run it and read the output. Click the play button or press <b>Shift + Enter</b>. Go top to bottom. If something breaks, go to <b>Runtime → Restart and run all</b>.<br><br>
<b>Table of contents</b> — Click the list icon on the left sidebar to jump between sections.<br>
<b>Two cell types</b> — Code cells have a play button. Text cells (like this one) have instructions.<br>
<b>Run order matters</b> — Skipping a cell may cause errors in later ones.
</div>

<div>
<h3>What is Structured Data Extraction?</h3>

<p>In Lab 1, the AI wrote back in whatever format it felt like — like a blank page. Every response looked a little different.</p>

<p>Maria texts 311: <em>"There's a broken fridge and old mattresses dumped behind my building at 123 Oak Ave."</em> Someone has to read that, figure out the location, the waste type, and the urgency, and type each one into a different field. Every time. For every report.</p>

<p>Structured Data Extraction is like giving the AI a <strong>form to fill in</strong> instead of a blank page. You define the exact fields you want, and the AI fills them in the same way every time — guaranteed. A 311 database cannot parse a paragraph. It can read a form.</p>

<p><strong>Labs 1–3 progression:</strong></p>
<table style="border-collapse:collapse;">
  <tbody>
    <tr style="background-color:#fafafa;"><td style="border:1px solid #ccc; padding:8px 16px;">Lab 1</td><td style="border:1px solid #ccc; padding:8px 16px;">AI responds to prompts — and you control how</td></tr>
    <tr style="background-color:#f0f4ff; font-weight:bold;"><td style="border:1px solid #ccc; padding:8px 16px;">Lab 2 ← you are here</td><td style="border:1px solid #ccc; padding:8px 16px;">AI extracts structured data from messy text</td></tr>
    <tr style="background-color:#fafafa;"><td style="border:1px solid #ccc; padding:8px 16px;">Lab 3</td><td style="border:1px solid #ccc; padding:8px 16px;">AI reads images and generates civic reports</td></tr>
  </tbody>
</table>
</div>

# Setting Up Your Development Environment
- Install the Google Generative AI Python library.

In [1]:
!pip install -q google-generativeai

# Importing and Initializing Gemini

Store your API key in Colab Secrets:
 1. Click the key icon in the left sidebar
 2. Add a secret named `GEMINI_API_KEY`
 3. Paste your API key as the value
 4. Toggle "Notebook access" to ON

Get your free API key at: https://aistudio.google.com

> **Note:** You may see a warning that `google.generativeai` is deprecated. This is expected — the code still works correctly. Ignore the warning and continue.

In [3]:
import google.generativeai as genai
from google.colab import userdata
import json
import time

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

Gemini initialized successfully.


# Part 1: The Problem — Inconsistent Output

A resident sends this message to the San Jose 311 system:

> *"There's a bunch of old furniture and a broken fridge dumped behind my building at 123 Oak Ave."*

Ask the model to extract the key information — no schema, plain text.

Run the cell a few times and notice how the format changes each time.
A 311 database cannot reliably read output that looks different on every call.

In [4]:
resident_message = (
    "There's a bunch of old furniture and a broken fridge "
    "dumped behind my building at 123 Oak Ave."
)

def extract_unstructured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction="Extract the location, waste type, and urgency from this illegal dumping report."
    )
    response = m.generate_content(message)
    time.sleep(12)  # stays under free tier rate limit
    return response.text

print("--- Resident message ---")
print(resident_message)
print("\n--- Unstructured extraction (run 3 times — notice the format changes) ---")
print("\nRun 1:")
print(extract_unstructured(resident_message))
print("\nRun 2:")
print(extract_unstructured(resident_message))
print("\nRun 3:")
print(extract_unstructured(resident_message))

--- Resident message ---
There's a bunch of old furniture and a broken fridge dumped behind my building at 123 Oak Ave.

--- Unstructured extraction (run 3 times — notice the format changes) ---

Run 1:
*   **Location:** 123 Oak Ave (behind the building)
*   **Waste Type:** Old furniture, broken fridge
*   **Urgency:** Not explicitly stated (it's a report of an existing dump, not flagged as an immediate hazard)

Run 2:
*   **Location:** Behind 123 Oak Ave.
*   **Waste Type:** Old furniture and a broken fridge.
*   **Urgency:** Moderate (implied, as it's already dumped and needs removal, but not explicitly stated as hazardous or immediate).

Run 3:
*   **Location:** Behind 123 Oak Ave
*   **Waste Type:** Old furniture, broken fridge
*   **Urgency:** Not explicitly stated (standard illegal dumping report, no immediate hazard mentioned).


# Part 2: Define a Schema

A schema is the form we give the AI to fill in.

We define five fields — `location`, `waste_type`, `urgency`, `department`, and `resident_language` — and tell the AI exactly what type each field should be.

The AI cannot return anything outside this shape.

Notice the two new fields compared to a basic 3-field schema:
- **`department`** — which San Jose city agency should handle this report
- **`resident_language`** — what language the resident wrote in (important for follow-up communication)

In [5]:
# The schema — the form the AI must fill in exactly.
schema_prompt = """
Extract information from this 311 resident complaint.
Return ONLY valid JSON with exactly these five fields:
{
  "location": string (the street address or location described),
  "waste_type": string (the type of items or problem described),
  "urgency": "LOW" or "MEDIUM" or "HIGH",
  "department": string (which San Jose city department should respond: e.g. "Environmental Services", "Public Works", "Code Enforcement", "Parks & Recreation"),
  "resident_language": string (language the resident wrote in, e.g. "English", "Spanish", "Vietnamese")
}
Urgency guide: LOW = small litter or cosmetic issue, MEDIUM = large items or ongoing problem, HIGH = hazardous materials or immediate safety risk.
No explanation. No markdown. JSON only.
"""

print("Schema defined. The AI must return exactly these five fields:")
for field in ["location", "waste_type", "urgency", "department", "resident_language"]:
    print(f"  - {field}")

Schema defined. The AI must return exactly these five fields:
  - location
  - waste_type
  - urgency
  - department
  - resident_language


# Part 3: Extract with the Schema

Now use the schema as the system instruction — the key change from Part 1.

Run it three times on the same message. The format will be identical every time.

That consistency is what makes the output safe to send to a 311 database.

In [6]:
def extract_structured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )
    response = m.generate_content(message)
    time.sleep(12)  # stays under free tier rate limit
    raw = response.text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

print("--- Resident message ---")
print(resident_message)
print("\n--- Structured extraction (run 3 times — format is identical each time) ---")

for i in range(1, 4):
    print(f"\nRun {i}:")
    result = extract_structured(resident_message)
    print(json.dumps(result, indent=2, ensure_ascii=False))

--- Resident message ---
There's a bunch of old furniture and a broken fridge dumped behind my building at 123 Oak Ave.

--- Structured extraction (run 3 times — format is identical each time) ---

Run 1:
{
  "location": "123 Oak Ave",
  "waste_type": "old furniture and a broken fridge",
  "urgency": "MEDIUM",
  "department": "Environmental Services",
  "resident_language": "English"
}

Run 2:
{
  "location": "123 Oak Ave",
  "waste_type": "Old furniture and broken fridge",
  "urgency": "MEDIUM",
  "department": "Environmental Services",
  "resident_language": "English"
}

Run 3:
{
  "location": "123 Oak Ave",
  "waste_type": "old furniture and a broken fridge",
  "urgency": "MEDIUM",
  "department": "Environmental Services",
  "resident_language": "English"
}


# Part 4: Test Three Different Complaint Types

Real 311 systems receive hundreds of different complaint types.
Here are three messages that represent very different situations.

Run the cell and observe:
- Does the AI assign the right department for each?
- Does it detect the resident's language correctly?
- Where does it seem confident? Where does it seem uncertain?

Pay attention to the **homeless encampment complaint** in particular.

In [ ]:
test_messages = [
    {
        "label": "Pothole",
        "message": "There's a massive pothole on Alum Rock Ave near Story Rd. "
                   "It's been there for months and I almost blew my tire yesterday."
    },
    {
        "label": "Streetlight (Spanish)",
        "message": "La luz de la calle en la esquina de la calle Oak y la avenida King "
                   "lleva semanas apagada. Es muy peligroso por la noche."
    },
    {
        "label": "Homeless encampment",
        "message": "There is a homeless encampment behind the Safeway on Capitol Ave. "
                   "There's trash and what looks like used needles nearby. "
                   "I'm worried about safety for my kids."
    }
]

for item in test_messages:
    print(f"=== {item['label']} ===")
    print(f"Input: {item['message']}")
    result = extract_structured(item["message"])
    print("Output:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print()

=== Pothole ===
Input: There's a massive pothole on Alum Rock Ave near Story Rd. It's been there for months and I almost blew my tire yesterday.
Output:
{
  "location": "Alum Rock Ave near Story Rd",
  "waste_type": "Pothole",
  "urgency": "HIGH",
  "department": "Public Works",
  "resident_language": "English"
}

=== Streetlight (Spanish) ===
Input: La luz de la calle en la esquina de la calle Oak y la avenida King lleva semanas apagada. Es muy peligroso por la noche.
Output:
{
  "location": "Oak Street and King Avenue corner",
  "waste_type": "Street light out",
  "urgency": "HIGH",
  "department": "Public Works",
  "resident_language": "Spanish"
}

=== Homeless encampment ===
Input: There is a homeless encampment behind the Safeway on Capitol Ave. There's trash and what looks like used needles nearby. I'm worried about safety for my kids.
Output:
{
  "location": "behind the Safeway on Capitol Ave.",
  "waste_type": "homeless encampment with trash and used needles",
  "urgency": "HIG

# Part 5: Discussion and Reflection

**Write your answers directly in this text cell. Double-click to edit it.**

---

**Question 1 — Classification consequences:**
The AI classified the homeless encampment complaint under a specific department.
Look at what it assigned. Does that seem right? If the AI misclassified this
report — sent it to the wrong department — what would actually happen to the
resident's complaint? Who pays the price for that error?

*Your answer here:*
The AI classified the homeless encampment under Public Works with a HIGH level of urgency, which is incorrect because Public Works handles infrastructure, such as potholes and broken streetlights, rather than people. If this report were actually sent to Public Works, the result would be that a sanitation crew shows up, clears the homeless encampments, and leaves. Although the trash would be cleaned out of the area, the people involved are simply pushed to a new location without any aid or connection to housing services. The price paid for the error would be the homeless people who had their belongings destroyed.


---

**Question 2 — Who designs the categories:**
The `department` field forces the AI to pick from a predefined list.
But who decided what departments exist, and what falls under each one?
Think of a complaint that doesn't fit neatly into any standard 311 category.
What happens to it?

*Your answer here:*
The departments in the predefined list, such as the Environmental Services, Public Works, Code Enforcement, and Parks & Recreation, were chosen by whoever designed the system. This is typically a small group of city employees and IT workers, rather than the actual residents who are filing the complaints. The categories already have a built-in bias toward treating everything like a physical problem that needs to be fixed and cleaned up. However, not every complaint fits into such predetermined categories. What if someone reports a noise complaint, but the real issue is that their neighbor is going through a mental health crisis? Or someone files a report about a "suspicious person" who, in reality, is just a homeless person living in the area? The AI would then file the complaint into the category that best fits, when the reality is that the likelihood of it actually fitting is slim to none. So the complaint ends up being sent to whichever department handles the surface-level problem, such as noise or trash, rather than the department that could actually address what's really going on. The complaint gets marked as resolved in the system, but the actual problem never gets fixed.
---

**Question 3 — Structured vs. unstructured:**
In Part 1 (unstructured), the AI had freedom. In Part 3 (structured), it was constrained.
What did the constraint gain? What did it lose? In what situations would
you want less structure, not more?

*Your answer here:*
The constraint is reliable because it returns the same JSON format every time and can be loaded directly into a database without any extra work. However, it misses the minute details needed. In Part 1, where the output was unstructured, the AI would include notes that result in more context. On the other hand, the JSON version removes such details since there's no place for them in the schema. Additionally, it forces the AI to pick a definitive answer rather than suggesting a human to address it. Less structure would be better when doing analysis,  such as reading through complaints to understand what residents actually care about.